In [1]:
import re
import matplotlib.pyplot as plt

In [2]:
from tokenizer import A1Tokenizer, PATTERN
from collections import Counter
import torch

tokenizer = A1Tokenizer.from_file("tokenizers/tokenizer_16384.pkl")

print("Vocabulary size:", len(tokenizer))
print("Number of merge rules:", len(tokenizer.merge_rules))
print("Max model length:", tokenizer.model_max_length)

print("\nSpecial token IDs:")
print("PAD:", tokenizer.pad_token_id)
print("UNK:", tokenizer.unk_token_id)
print("BOS:", tokenizer.bos_token_id)
print("EOS:", tokenizer.eos_token_id)

Vocabulary size: 16384
Number of merge rules: 16124
Max model length: 256

Special token IDs:
PAD: 0
UNK: 1
BOS: 2
EOS: 3


In [3]:
print("First 20 merge rules:\n")

for rank, pair in enumerate(tokenizer.merge_rules[:20]):
    print(
        f"{rank:4d}: {pair[0]!r} + {pair[1]!r} "
        f"-> {(pair[0] + pair[1])!r}"
    )

print("\nLast 20 merge rules:\n")

start = max(0, len(tokenizer.merge_rules) - 20)

for rank in range(start, len(tokenizer.merge_rules)):
    pair = tokenizer.merge_rules[rank]

    print(
        f"{rank:4d}: {pair[0]!r} + {pair[1]!r} "
        f"-> {(pair[0] + pair[1])!r}"
    )

First 20 merge rules:

   0: b' ' + b't' -> b' t'
   1: b'h' + b'e' -> b'he'
   2: b' ' + b'a' -> b' a'
   3: b'i' + b'n' -> b'in'
   4: b' t' + b'he' -> b' the'
   5: b'e' + b'r' -> b'er'
   6: b'o' + b'n' -> b'on'
   7: b'r' + b'e' -> b're'
   8: b' ' + b'o' -> b' o'
   9: b'a' + b't' -> b'at'
  10: b' ' + b's' -> b' s'
  11: b'e' + b'n' -> b'en'
  12: b'e' + b'd' -> b'ed'
  13: b'a' + b'n' -> b'an'
  14: b' ' + b'c' -> b' c'
  15: b'i' + b's' -> b'is'
  16: b' ' + b'w' -> b' w'
  17: b'i' + b't' -> b'it'
  18: b'e' + b's' -> b'es'
  19: b'o' + b'r' -> b'or'

Last 20 merge rules:

16104: b' exter' + b'min' -> b' extermin'
16105: b' B' + b'ordeaux' -> b' Bordeaux'
16106: b' J' + b'oshua' -> b' Joshua'
16107: b'C' + b'atal' -> b'Catal'
16108: b'B' + b'arn' -> b'Barn'
16109: b' \xc3' + b'\x81' -> b' \xc3\x81'
16110: b' el' + b'ves' -> b' elves'
16111: b' Charl' + b'ton' -> b' Charlton'
16112: b'Ec' + b'uador' -> b'Ecuador'
16113: b' sens' + b'ation' -> b' sensation'
16114: b' meaning' +

In [4]:
text = "She lives in Göteborg, Sweden in 2026!"

chunks = tokenizer.pattern.findall(text.strip())

print("Original:", repr(text))
print("\nPre-tokenized chunks:")

for i, chunk in enumerate(chunks):
    print(f"{i:2d}: {chunk!r}")

print("\nReconstruction:", repr("".join(chunks)))
assert "".join(chunks) == text.strip()

Original: 'She lives in Göteborg, Sweden in 2026!'

Pre-tokenized chunks:
 0: 'She'
 1: ' lives'
 2: ' in'
 3: ' Göteborg'
 4: ','
 5: ' Sweden'
 6: ' in'
 7: ' 2026'
 8: '!'

Reconstruction: 'She lives in Göteborg, Sweden in 2026!'


In [5]:
text = "She lives in Göteborg, Sweden in 2026!"

chunks = tokenizer.pattern.findall(text.strip())

print("Original:", text)

for chunk in chunks:
    ids = tokenizer._bpe(chunk)
    tokens = [tokenizer.int_to_token[i] for i in ids]

    print(f"\nChunk: {chunk!r}")
    print("Tokens:", tokens)
    print("IDs:   ", ids)

Original: She lives in Göteborg, Sweden in 2026!

Chunk: 'She'
Tokens: [b'She']
IDs:    [2645]

Chunk: ' lives'
Tokens: [b' lives']
IDs:    [5354]

Chunk: ' in'
Tokens: [b' in']
IDs:    [283]

Chunk: ' Göteborg'
Tokens: [b' G', b'\xc3\xb6', b'te', b'borg']
IDs:    [403, 2867, 593, 13735]

Chunk: ','
Tokens: [b',']
IDs:    [48]

Chunk: ' Sweden'
Tokens: [b' Sweden']
IDs:    [6589]

Chunk: ' in'
Tokens: [b' in']
IDs:    [283]

Chunk: ' 2026'
Tokens: [b' 20', b'26']
IDs:    [588, 5673]

Chunk: '!'
Tokens: [b'!']
IDs:    [37]


In [6]:
words = [
    "the",
    "and",
    "language",
    "transformer",
    "international",
    "electrochemistry",
    "Göteborg",
    "123456789"
]

for word in words:
    ids = tokenizer._bpe(word)
    tokens = [tokenizer.int_to_token[i] for i in ids]

    print(f"{word:<22} -> {tokens}")
    print(f"{'':22}    {len(ids)} BPE tokens\n")

the                    -> [b'the']
                          1 BPE tokens

and                    -> [b'and']
                          1 BPE tokens

language               -> [b'language']
                          1 BPE tokens

transformer            -> [b'trans', b'form', b'er']
                          3 BPE tokens

international          -> [b'in', b'ternational']
                          2 BPE tokens

electrochemistry       -> [b'elect', b'roc', b'hemistry']
                          3 BPE tokens

Göteborg               -> [b'G', b'\xc3\xb6', b'te', b'borg']
                          4 BPE tokens

123456789              -> [b'12', b'34', b'5', b'67', b'89']
                          5 BPE tokens



In [7]:
texts = [
    "This is a test.",
    "She lives in Stockholm.",
    "Natural language processing is interesting.",
    "Göteborg is a city in Sweden."
]

for text in texts:
    encoded = tokenizer(text, padding=False, truncation=False)

    ids = encoded["input_ids"][0]
    decoded = tokenizer.decode(ids)

    print("\nOriginal:", text)
    print("Token IDs:", ids)
    print("Decoded:  ", decoded)

    assert decoded == text.strip()

print("\nAll encoding/decoding tests passed!")


Original: This is a test.
Token IDs: [2, 673, 319, 262, 1890, 50, 3]
Decoded:   This is a test.

Original: She lives in Stockholm.
Token IDs: [2, 2645, 5354, 283, 8475, 14286, 50, 3]
Decoded:   She lives in Stockholm.

Original: Natural language processing is interesting.
Token IDs: [2, 12459, 1281, 5652, 319, 12239, 50, 3]
Decoded:   Natural language processing is interesting.

Original: Göteborg is a city in Sweden.
Token IDs: [2, 75, 2867, 593, 13735, 319, 262, 1015, 283, 6589, 50, 3]
Decoded:   Göteborg is a city in Sweden.

All encoding/decoding tests passed!
